# Phase 2.3–2.6 — `analyse(video)`

The tool. Takes a video path, returns rallies and per-shot records conforming to `shot_record_schema.json`.

```
video.mp4
  -> activity gate      coarse detector pass, keeps regions with both players present and moving
  -> pose               RTMPose-l on player crops, active regions only
  -> canonicalise       hip-centred, torso-scaled, side-mirrored
  -> contact detection  per-frame probability, peak-picked with NMS
  -> classification     4-class + technique, temperature-calibrated
  -> kinematics         13 scalars per shot, derived from the stored pose window
  -> rally grouping     gaps in the contact sequence
```

### Calibration comes first, and it matters

The final classifier trained to a loss of 0.0042 with 95.7% mean confidence on data it memorised. Raw softmax from it is badly overconfident. Phase 3 gates coaching feedback on `class_confidence`, so an uncalibrated 0.96 on a wrong shot produces confidently wrong advice.

Temperature is fitted on **LOVO out-of-fold** probabilities — genuinely held-out — then applied to the final model.

### Rally gating — the design correction

Not "both players close to the table". **Defenders back away** — `test_1__right` is a chopper playing 2–3 m behind the endline with 28 of 40 strokes defensive. Gating on proximity would discard exactly the class the coach most needs to see. Gate on **activity**, keep table distance as a *feature*.

### Runtime

~2–4× realtime on a T4. The activity gate typically removes 40–60% of frames on match footage. Pose is cached to `derived/analysed/{video_id}/`, so re-running is near-instant.


## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Setup

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

ACT_STRIDE   = 8       # detector stride for the activity gate
ACT_PAD_S    = 1.0     # pad active regions by this many seconds
MOTION_THR   = 0.02    # torso-normalised box motion to count as "active"
DET_THR      = 0.60    # contact probability threshold
NMS_GAP      = 30      # min frames between contacts (0.25 s)
RALLY_GAP_S  = 2.5     # gap in the contact sequence that starts a new rally
ABSTAIN_Q    = 0.15    # abstain on the least-confident fraction

import json, math, shutil, time, os, glob, site
from pathlib import Path
import numpy as np, pandas as pd

_lp = Path("/content/_ort_libpath.txt")
if not _lp.exists():
    libs = []
    for sp in site.getsitepackages():
        libs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
    if libs: _lp.write_text(":".join(libs))
if _lp.exists():
    os.environ["LD_LIBRARY_PATH"] = _lp.read_text() + ":" + os.environ.get("LD_LIBRARY_PATH","")

import cv2, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

BASE = Path(BASE); META = BASE/"derived/meta"; CKPT = BASE/"models/checkpoints"
ANALYSED = BASE/"derived/analysed"; ANALYSED.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/_work"); LOCAL.mkdir(exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."

folds = json.loads((META/"folds.json").read_text())
PRE, NF, FPS = folds["window"]["pre"], folds["window"]["n_frames"], 120
CLASSES = ["serve","attack","control","defence"]
TECHS   = ["block","chop","flick","lob","loop","push","serve","smash"]

L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI = 5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK = 11,12,13,14,15,16
FLIP = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
print(f"{torch.cuda.get_device_name(0)}  |  window {NF}f, contact @ {PRE}")

Tesla T4  |  window 97f, contact @ 60


In [4]:
# =============================================================================
# CELL 1b — INSTALL  (run once per fresh runtime, then RESTART the session)
#
# rtmlib declares CPU onnxruntime as a hard dependency, which shadows
# onnxruntime-gpu and silently drops RTMPose to CPU. Install it --no-deps
# and put the GPU wheel LAST.
# =============================================================================

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q opencv-python numpy tqdm ultralytics pyarrow 2>&1 | tail -1
!pip install -q "onnxruntime-gpu==1.22.0" 2>&1 | tail -1

print("\ninstalled (must show ONLY onnxruntime-gpu):")
!pip list 2>/dev/null | grep -iE "onnxruntime|rtmlib|ultralytics"

# expose torch's bundled CUDA libs so onnxruntime-gpu can find libcudnn
import os, glob, site
libs = []
for sp in site.getsitepackages():
    libs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
if libs:
    open("/content/_ort_libpath.txt", "w").write(":".join(libs))
    print(f"\nsaved {len(libs)} nvidia lib paths")

print("\n" + "="*60)
print("  NOW: Runtime > Restart session, then run from cell 2.")
print("="*60)

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.4 MB/s eta 0:00:00

installed (must show ONLY onnxruntime-gpu):
onnxruntime-gpu                       1.22.0
rtmlib                                0.0.16
ultralytics                           8.4.126
ultralytics-platform                  0.1.11
ultralytics-thop                      2.1.6

saved 17 nvidia lib paths

  NOW: Runtime > Restart session, then run from cell 2.


## 3 · Architectures & checkpoints

In [5]:
class Block(nn.Module):
    def __init__(s,c,d,drop=0.1):
        super().__init__()
        s.c1=nn.Conv1d(c,c,5,padding=2*d,dilation=d); s.c2=nn.Conv1d(c,c,5,padding=2*d,dilation=d)
        s.n1,s.n2=nn.BatchNorm1d(c),nn.BatchNorm1d(c); s.do=nn.Dropout(drop)
    def forward(s,x):
        r=x; x=s.do(F.gelu(s.n1(s.c1(x)))); x=s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x+r)

class DetNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d) for d in (1,2,4,8,16,32,64)])
        s.hc,s.hs=nn.Conv1d(w,1,1),nn.Conv1d(w,1,1)
    def forward(s,x):
        z=s.blocks(s.stem(x)); return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s,c):
        super().__init__(); s.score=nn.Conv1d(c,1,1)
    def forward(s,x):
        w=torch.softmax(s.score(x),-1)
        return torch.cat([(x*w).sum(-1), x.max(-1).values],-1)

class ClsNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d,0.2) for d in (1,2,4,8,16,32)])
        s.pool=AttnPool(w)
        s.trunk=nn.Sequential(nn.Linear(w*2,256),nn.GELU(),nn.Dropout(0.3))
        s.shot,s.tech=nn.Linear(256,4),nn.Linear(256,8)
    def forward(s,x):
        z=s.trunk(s.pool(s.blocks(s.stem(x)))); return s.shot(z), s.tech(z)

dck = torch.load(CKPT/"detector_final.pt", map_location=dev, weights_only=False)
cck = torch.load(CKPT/"classifier_final.pt", map_location=dev, weights_only=False)
DET = DetNet(dck["c_in"]).to(dev); DET.load_state_dict(dck["state"]); DET.eval()
CLS = ClsNet(cck["c_in"]).to(dev); CLS.load_state_dict(cck["state"]); CLS.eval()
DMU, DSD = dck["mu"], dck["sd"]
CMU, CSD = cck["mu"].to(dev), cck["sd"].to(dev)
print(f"  detector   {dck['c_in']} ch   LOVO F1 {dck['lovo_f1_at_tol8']}")
print(f"  classifier {cck['c_in']} ch   LOVO macro-F1 {cck['lovo_macro_f1']}")

from ultralytics import YOLO
from rtmlib import RTMPose
YOLO_DET = YOLO(str(BASE/"models/detector/best.pt")); YOLO_DET.to("cuda")
PLAYER_CLS = [k for k,v in YOLO_DET.names.items() if v.lower()=="player"][0]
TABLE_CLS  = [k for k,v in YOLO_DET.names.items() if v.lower()=="table"][0]
POSE = RTMPose(onnx_model=("https://download.openmmlab.com/mmpose/v1/projects/"
        "rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
        "3f5a1437_20230504.zip"), model_input_size=(288,384),
        backend="onnxruntime", device="cuda")
print("  pose + detector ready")

  detector   170 ch   LOVO F1 0.881
  classifier 86 ch   LOVO macro-F1 0.792
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip
100%|██████████| 98.9M/98.9M [00:03<00:00, 28.8MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
  pose + detector ready


## 4 · Calibration (step 2.5)

A single temperature scalar fitted on 7-fold LOVO out-of-fold probabilities from `canonical.npz`. Those predictions are genuinely held-out, so the temperature they imply is honest even though the final model has no validation set of its own.

Reliability is reported before and after: **ECE** is the average gap between stated confidence and actual accuracy. An uncalibrated model that says 0.96 and is right 0.79 of the time has ECE ≈ 0.17, and that gap is exactly what would mislead Phase 3.

In [6]:
CAL_PATH = META/"calibration.json"

def ece(conf, correct, bins=10):
    e = 0.0
    for lo in np.linspace(0, 1, bins+1)[:-1]:
        m = (conf >= lo) & (conf < lo+1/bins)
        if m.sum(): e += m.mean()*abs(correct[m].mean() - conf[m].mean())
    return float(e)

if CAL_PATH.exists():
    TEMP = json.loads(CAL_PATH.read_text())["temperature"]
    print(f"loaded temperature {TEMP:.3f}")
else:
    d = np.load(BASE/"derived/clips/canonical.npz", allow_pickle=True)
    u = d["usable"]
    KP, VEL, VAL = d["kp"].astype(np.float32), d["vel"].astype(np.float32), d["valid"]
    TD = np.nan_to_num(d["table_dist"].astype(np.float32))
    Xc = np.concatenate([KP[:,0].reshape(len(u),NF,-1),
                         VEL[:,0].reshape(len(u),NF,-1),
                         VAL[:,0].astype(np.float32), TD[...,None]], -1)
    Xc = np.nan_to_num(Xc).transpose(0,2,1)[u]
    yc = pd.Series(d["shot_class"][u]).map({c:i for i,c in enumerate(CLASSES)}).values
    tc = pd.Series(d["technique"][u]).map({t:i for i,t in enumerate(TECHS)}).fillna(-1).astype(int).values
    fc = d["fold"][u]

    def focal(lg,tg,w=None,g=2.0):
        m=tg>=0
        if m.sum()==0: return lg.sum()*0.
        lg,tg=lg[m],tg[m]
        ce=F.cross_entropy(lg,tg,weight=w,reduction="none")
        pt=torch.exp(-F.cross_entropy(lg,tg,reduction="none"))
        return ((1-pt)**g*ce).mean()

    oof = np.zeros((len(yc),4), np.float32)
    for f in sorted(set(fc)):
        tr, va = fc!=f, fc==f
        xt=torch.tensor(Xc[tr],device=dev)
        mu,sd=xt.mean((0,2),keepdim=True),xt.std((0,2),keepdim=True)+1e-6
        xt=(xt-mu)/sd
        yt=torch.tensor(yc[tr],device=dev); tt=torch.tensor(tc[tr],device=dev)
        cnt=np.bincount(yc[tr],minlength=4).clip(1)
        cw=torch.tensor(len(yc[tr])/(4*cnt),dtype=torch.float32,device=dev)
        p=(1./cnt)[yc[tr]]; p=p/p.sum()
        net=ClsNet(Xc.shape[1]).to(dev)
        opt=torch.optim.AdamW(net.parameters(),lr=2e-3,weight_decay=1e-4)
        steps=60*max(1,math.ceil(tr.sum()/64))
        sch=torch.optim.lr_scheduler.OneCycleLR(opt,2e-3,total_steps=steps)
        net.train()
        for _ in range(steps):
            b=np.random.choice(tr.sum(),64,p=p)
            xb=xt[b]
            sh=torch.randint(-6,7,(64,),device=dev)
            ix=(torch.arange(NF,device=dev)[None]+sh[:,None]).clamp(0,NF-1)
            xb=torch.gather(xb,2,ix[:,None].expand(-1,xb.shape[1],-1))
            s_,t_=net(xb+torch.randn_like(xb)*0.01)
            loss=focal(s_,yt[b],cw)+0.2*focal(t_,tt[b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(),1.0); opt.step(); sch.step()
        net.eval()
        with torch.no_grad():
            lo,_=net((torch.tensor(Xc[va],device=dev)-mu)/sd)
            oof[va]=lo.cpu().numpy()
        print(f"  fold {f} done")

    lg = torch.tensor(oof); tg = torch.tensor(yc)
    logT = torch.zeros(1, requires_grad=True)
    o = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
    def closure():
        o.zero_grad()
        l = F.cross_entropy(lg/torch.exp(logT), tg); l.backward(); return l
    o.step(closure)
    TEMP = float(torch.exp(logT).item())

    p0 = torch.softmax(lg,1).numpy(); p1 = torch.softmax(lg/TEMP,1).numpy()
    corr = (p0.argmax(1)==yc).astype(float)
    e0, e1 = ece(p0.max(1),corr), ece(p1.max(1),corr)
    CAL_PATH.write_text(json.dumps({"temperature":TEMP,"ece_before":e0,
        "ece_after":e1,"accuracy":float(corr.mean()),
        "mean_conf_before":float(p0.max(1).mean()),
        "mean_conf_after":float(p1.max(1).mean())}, indent=2))
    print(f"\n  temperature {TEMP:.3f}")
    print(f"  ECE {e0:.4f} -> {e1:.4f}")
    print(f"  mean confidence {p0.max(1).mean():.3f} -> {p1.max(1).mean():.3f}"
          f"   (accuracy {corr.mean():.3f})")

cal = json.loads(CAL_PATH.read_text())
print(f"\nTEMP={TEMP:.3f}  — confidences are divided by this before softmax")

  fold A done
  fold B done
  fold C done
  fold D done
  fold E done
  fold F done
  fold G done

  temperature 2.202
  ECE 0.1255 -> 0.0088
  mean confidence 0.892 -> 0.758   (accuracy 0.766)

TEMP=2.202  — confidences are divided by this before softmax


## 5 · Video → pose, with an activity gate

The gate is the cheap part: the player detector at stride 8 marks frames where **both players are present and at least one is moving**. Pose then runs only inside those regions.

That is also the rally gate — and note it keys on *motion*, not table proximity, so a chopper standing 3 m back is still inside it.

In [7]:
def resolve(frames, mid_x, conf=0.35):
    out=[]
    for r in YOLO_DET.predict(frames, verbose=False, conf=conf):
        d={"left":None,"right":None}
        if r.boxes is not None and len(r.boxes):
            xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
            pl=xy[cl==PLAYER_CLS]
            if len(pl):
                cx=(pl[:,0]+pl[:,2])/2
                ls,rs=pl[cx<mid_x],pl[cx>=mid_x]
                if len(ls): d["left"]=ls[np.argmin((ls[:,0]+ls[:,2])/2)]
                if len(rs): d["right"]=rs[np.argmax((rs[:,0]+rs[:,2])/2)]
        out.append(d)
    return out

def find_table(cap,n,k=9):
    bx=[]
    for f in np.linspace(n*0.1,n*0.9,k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,fr=cap.read()
        if not ok: continue
        r=YOLO_DET.predict(fr,verbose=False,conf=0.35)[0]
        if r.boxes is None or not len(r.boxes): continue
        xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
        tb=xy[cl==TABLE_CLS]
        if len(tb): bx.append(tb[np.argmax((tb[:,2]-tb[:,0])*(tb[:,3]-tb[:,1]))])
    return np.median(np.stack(bx),0).astype(np.float32) if bx else None

def activity_gate(path, nfr, mid_x, stride=ACT_STRIDE):
    """-> merged (start,end) spans where both players are present and moving."""
    cap=cv2.VideoCapture(str(path))
    idx=list(range(0,nfr,stride)); flags=np.zeros(len(idx),bool)
    prev=None
    for k in tqdm(range(0,len(idx),64), desc="  activity", leave=False):
        chunk=idx[k:k+64]; frames=[]
        for f in chunk:
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,fr=cap.read()
            frames.append(fr if ok else np.zeros((720,1280,3),np.uint8))
        for j,d in enumerate(resolve(frames, mid_x)):
            both = d["left"] is not None and d["right"] is not None
            moving=True
            if both and prev is not None:
                h=max(d["left"][3]-d["left"][1],1)
                mv=max(abs((d["left"][0]+d["left"][2])/2-prev[0]),
                       abs((d["right"][0]+d["right"][2])/2-prev[1]))/h
                moving = mv > MOTION_THR
            if both:
                prev=((d["left"][0]+d["left"][2])/2,(d["right"][0]+d["right"][2])/2)
            flags[k+j]= both and moving
    cap.release()
    pad=int(ACT_PAD_S*FPS); spans=[]
    for k,f in enumerate(flags):
        if not f: continue
        s,e=max(0,idx[k]-pad),min(nfr-1,idx[k]+pad)
        if spans and s<=spans[-1][1]+1: spans[-1][1]=max(spans[-1][1],e)
        else: spans.append([s,e])
    return spans

def extract_pose(path, spans, mid_x, nfr):
    total=sum(e-s+1 for s,e in spans)
    F_,KP,SC,BX,DT,SG = (np.zeros(total,np.int32), np.zeros((total,2,17,2),np.float16),
        np.zeros((total,2,17),np.float16), np.zeros((total,2,4),np.float16),
        np.zeros((total,2),bool), np.zeros(total,np.int32))
    cap=cv2.VideoCapture(str(path)); w=0
    pb=tqdm(total=total, desc="  pose", leave=False)
    for si,(s0,e0) in enumerate(spans):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(s0)); pos=s0
        while pos<=e0:
            n=min(600,e0-pos+1); frames=[]
            for _ in range(n):
                ok,fr=cap.read()
                frames.append(fr if ok else (frames[-1] if frames else np.zeros((720,1280,3),np.uint8)))
            n=len(frames)
            di=list(range(0,n,8));  di+= [] if di[-1]==n-1 else [n-1]
            dets=resolve([frames[i] for i in di], mid_x)
            boxes={}
            for pi,side in enumerate(["left","right"]):
                kn=[(i,b) for i,b in zip(di,[d[side] for d in dets]) if b is not None]
                if not kn: continue
                ki=np.array([a for a,_ in kn],float); kb=np.stack([b for _,b in kn]).astype(float)
                boxes[pi]=np.stack([np.interp(np.arange(n),ki,kb[:,c]) for c in range(4)],1).astype(np.float32)
            for k in range(n):
                bb,who=[],[]
                for pi in (0,1):
                    if pi in boxes:
                        b=boxes[pi][k]; bw,bh=b[2]-b[0],b[3]-b[1]
                        b=np.array([max(0,b[0]-bw*.18),max(0,b[1]-bh*.11),
                                    b[2]+bw*.18,b[3]+bh*.045],np.float32)
                        bb.append(b); who.append(pi); BX[w+k,pi]=b; DT[w+k,pi]=True
                if bb:
                    kp,sc=POSE(frames[k],bboxes=np.stack(bb))
                    for j,pi in enumerate(who): KP[w+k,pi]=kp[j]; SC[w+k,pi]=sc[j]
                F_[w+k]=pos+k; SG[w+k]=si
            w+=n; pos+=n; pb.update(n); del frames
    pb.close(); cap.release()
    return dict(frame_idx=F_[:w],seg_id=SG[:w],keypoints=KP[:w],
                scores=SC[:w],boxes=BX[:w],detected=DT[:w])

print("extraction ready")

extraction ready


## 6 · Canonicalise, detect, classify, measure

In [8]:
def canon(kp,sc,seg,mirror):
    kp=kp.astype(np.float32).copy()
    hip=(kp[:,L_HIP]+kp[:,R_HIP])/2; sho=(kp[:,L_SHO]+kp[:,R_SHO])/2
    torso=np.linalg.norm(sho-hip,axis=-1); scale=np.ones(len(kp),np.float32)
    for s in np.unique(seg):
        m=seg==s; t=torso[m]; t=t[t>1]
        scale[m]=np.median(t) if len(t) else 1.
    kp=(kp-hip[:,None,:])/np.maximum(scale,1e-3)[:,None,None]
    if mirror:
        kp[...,0]*=-1; sc=sc.copy()
        for a,b in FLIP: kp[:,[a,b]]=kp[:,[b,a]]; sc[:,[a,b]]=sc[:,[b,a]]
    return kp,sc,scale

def build_stream(raw, table):
    seg=raw["seg_id"]; KP=raw["keypoints"]; SC=raw["scores"]; DT=raw["detected"]
    ch,kps,vals,tds=[],[],[],[]
    for pi in (0,1):
        kp,sc,scale=canon(KP[:,pi],SC[:,pi].astype(np.float32),seg,mirror=(pi==1))
        vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
        vel[np.diff(seg,prepend=seg[0])!=0]=0
        ch+=[kp.reshape(len(kp),-1),vel.reshape(len(kp),-1),
             (sc*DT[:,pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc>=.35)&DT[:,pi:pi+1])
        hx=(KP[:,pi,L_HIP,0]+KP[:,pi,R_HIP,0])/2
        edge=table[0] if pi==0 else table[2]
        tds.append(np.abs(hx-edge)/np.maximum(scale,1e-3) if table is not None and table[2]>table[0]
                   else np.zeros(len(kp),np.float32))
    cuts=np.where(np.diff(seg)!=0)[0]+1; b=np.concatenate([[0],cuts,[len(seg)]])
    return dict(X=np.nan_to_num(np.concatenate(ch,1).astype(np.float32)),
                kp=np.stack(kps,1), val=np.stack(vals,1), td=np.nan_to_num(np.stack(tds,1)),
                fidx=raw["frame_idx"], scores=SC, detected=DT,
                spans=[(int(b[i]),int(b[i+1])) for i in range(len(b)-1)])

def decode(prob,thr=DET_THR,gap=NMS_GAP):
    idx=np.where(prob>=thr)[0]
    if not len(idx): return np.array([],int)
    pk=[i for i in idx if prob[i]==prob[max(0,i-gap//2):i+gap//2+1].max()]
    pk=sorted(pk,key=lambda i:-prob[i]); keep=[]
    for p in pk:
        if all(abs(p-k)>=gap for k in keep): keep.append(p)
    return np.array(sorted(keep),int)

def window_at(st,idx,side):
    sel=np.clip(np.arange(idx-PRE,idx-PRE+NF),0,len(st["X"])-1)
    pi=0 if side=="left" else 1
    kp=st["kp"][sel,pi]; val=st["val"][sel,pi].astype(np.float32)
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    x=np.concatenate([kp.reshape(NF,-1),vel.reshape(NF,-1),val,st["td"][sel,pi][:,None]],1)
    return np.nan_to_num(x).T.astype(np.float32), kp, val

def angle(a,b,c):
    v1,v2=a-b,c-b
    cs=(v1*v2).sum(-1)/np.maximum(np.linalg.norm(v1,axis=-1)*np.linalg.norm(v2,axis=-1),1e-6)
    return np.degrees(np.arccos(np.clip(cs,-1,1)))

def kinematics(kp,val,td_win,wri):
    """13 scalars derived from the stored canonical window."""
    h=-kp[...,1]
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    spd=np.linalg.norm(vel,axis=-1); spd[~val.astype(bool)]=np.nan
    w=spd[:,wri]; pre,post=slice(0,PRE),slice(PRE,NF)
    sh,el=(R_SHO,R_ELB) if wri==R_WRI else (L_SHO,L_ELB)
    d_hip=np.linalg.norm(kp[:,wri],axis=-1)
    ea=angle(kp[:,sh],kp[:,el],kp[:,wri])
    pk=np.nanmax(w) if np.isfinite(w).any() else np.nan
    rec=np.nan
    if np.isfinite(pk) and pk>0:
        after=np.where(np.nan_to_num(w[post])<0.2*pk)[0]
        rec=float(after[0]) if len(after) else np.nan
    trunk=(kp[:,L_SHO]+kp[:,R_SHO])/2
    ta=np.degrees(np.arctan2(trunk[:,0],-trunk[:,1]))
    return {
        "backswing_amplitude": float(np.nanmax(d_hip[pre])),
        "peak_wrist_speed":    float(pk),
        "time_to_peak":        int(np.nanargmax(w)-PRE) if np.isfinite(w).any() else None,
        "contact_height":      float(h[PRE,wri]-(h[PRE,L_SHO]+h[PRE,R_SHO])/2),
        "elbow_angle":         float(ea[PRE]),
        "elbow_range":         float(np.nanmax(ea)-np.nanmin(ea)),
        "trunk_lean":          float(ta[PRE]),
        "trunk_rotation":      float(np.nanmax(ta)-np.nanmin(ta)),
        "table_distance":      float(td_win[PRE]),
        "stance_width":        float(abs(kp[PRE,L_ANK,0]-kp[PRE,R_ANK,0])),
        "knee_angle":          float((angle(kp[PRE:PRE+1,L_HIP],kp[PRE:PRE+1,L_KNE],kp[PRE:PRE+1,L_ANK])[0]+
                                      angle(kp[PRE:PRE+1,R_HIP],kp[PRE:PRE+1,R_KNE],kp[PRE:PRE+1,R_ANK])[0])/2),
        "follow_through":      float(np.nansum(w[post])),
        "recovery_time":       rec,
    }
print("ready")

ready


## 7 · `analyse()`

Rallies are formed by gaps in the **contact sequence**, not by table proximity. `RALLY_GAP_S = 2.5` — longer than any inter-stroke interval within a rally, shorter than a between-point pause.

In [9]:
def analyse(video_path, video_id=None, force=False, save_pose=True):
    video_path=Path(video_path); vid=video_id or video_path.stem
    outdir=ANALYSED/vid; outdir.mkdir(parents=True, exist_ok=True)
    cache=outdir/"pose_raw.npz"
    t0=time.time()

    local=LOCAL/video_path.name
    if not local.exists(): shutil.copy(video_path, local)
    cap=cv2.VideoCapture(str(local))
    nfr=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS)
    W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    table=find_table(cap,nfr); cap.release()
    mid_x=(table[0]+table[2])/2 if table is not None else W/2
    print(f"{vid}: {nfr:,} frames @ {fps:.0f}fps ({nfr/fps/60:.1f} min)")

    if cache.exists() and not force:
        raw={k:v for k,v in np.load(cache,allow_pickle=True).items()}
        print(f"  cached pose: {len(raw['frame_idx']):,} frames")
    else:
        spans=activity_gate(local,nfr,mid_x)
        cov=sum(e-s+1 for s,e in spans)/max(nfr,1)
        print(f"  active: {len(spans)} regions, {cov:.1%} of video")
        raw=extract_pose(local,spans,mid_x,nfr)
        if save_pose:
            np.savez_compressed(cache, **raw, table_box=table if table is not None else np.zeros(4,np.float32))
    st=build_stream(raw, table)

    prob=np.zeros(len(st["X"]),np.float32); sidep=np.zeros(len(st["X"]),np.float32)
    with torch.no_grad():
        for a,b in st["spans"]:
            x=torch.tensor(((st["X"][a:b]-DMU)/DSD).T[None],dtype=torch.float32,device=dev)
            lc,ls=DET(x)
            prob[a:b]=torch.sigmoid(lc)[0].cpu().numpy()
            sidep[a:b]=torch.sigmoid(ls)[0].cpu().numpy()
    peaks=decode(prob)
    print(f"  contacts: {len(peaks)}")
    if not len(peaks):
        return pd.DataFrame(), np.zeros((0,NF,17,2),np.float16)

    sides=["right" if sidep[i]>=0.5 else "left" for i in peaks]
    wins,kps,vals=[],[],[]
    for i,s in zip(peaks,sides):
        x,kp,val=window_at(st,i,s); wins.append(x); kps.append(kp); vals.append(val)
    with torch.no_grad():
        lo,lt=CLS((torch.tensor(np.stack(wins),device=dev)-CMU)/CSD)
        pr=torch.softmax(lo/TEMP,1).cpu().numpy()          # calibrated
        pt=torch.softmax(lt/TEMP,1).cpu().numpy()

    floor=float(np.quantile(pr.max(1),ABSTAIN_Q)) if len(pr)>3 else 0.0
    frames=st["fidx"][peaks]
    rid,shot_idx,cur,last=[],[],0,None
    for f in frames:
        if last is not None and (f-last)/FPS>RALLY_GAP_S: cur+=1; k=0
        else: k=0 if last is None else shot_idx[-1]+1
        rid.append(cur); shot_idx.append(k); last=f
    rlen=pd.Series(rid).value_counts().to_dict()

    rows=[]
    for n,(i,s) in enumerate(zip(peaks,sides)):
        pi=0 if s=="left" else 1
        km=kinematics(kps[n],vals[n],st["td"][np.clip(np.arange(i-PRE,i-PRE+NF),0,len(st["X"])-1),pi],
                      R_WRI if s=="left" else L_WRI)
        sel=np.clip(np.arange(i-PRE,i-PRE+NF),0,len(st["X"])-1)
        rows.append(dict(video_id=vid, rally_id=rid[n], shot_index=shot_idx[n],
            player=s, frame=int(frames[n]), timestamp_s=float(frames[n]/FPS),
            shot_class=CLASSES[int(pr[n].argmax())],
            class_confidence=float(pr[n].max()),
            class_proba=[float(z) for z in pr[n]],
            technique=TECHS[int(pt[n].argmax())],
            abstain=bool(pr[n].max()<floor),
            detect_confidence=float(prob[i]),
            rally_length=int(rlen[rid[n]]),
            pose_confidence=float(st["scores"][sel,pi][st["scores"][sel,pi]>0].mean()
                                  if (st["scores"][sel,pi]>0).any() else 0.),
            detected=float(st["detected"][sel,pi].mean()), **km))
    df=pd.DataFrame(rows)
    pose_windows=np.stack(kps).astype(np.float16)

    df.to_parquet(outdir/"shots.parquet", index=False)
    np.savez_compressed(outdir/"pose_windows.npz",
                        stroke_id=df.apply(lambda r:f"{vid}_{r.frame:07d}",axis=1).values.astype(str),
                        pose_window=pose_windows, valid=np.stack(vals))
    print(f"  {len(df)} shots, {df.rally_id.nunique()} rallies, "
          f"{df.abstain.sum()} abstained  [{time.time()-t0:.0f}s, "
          f"{(time.time()-t0)/(nfr/fps):.1f}x realtime]")
    return df, pose_windows

print("analyse() ready")

analyse() ready


## 8 · Validate on a known video

`game_1` has ground truth, so shot and rally counts can be checked. **This is not an accuracy measurement** — `game_1` was in the final models' training set. It verifies the pipeline runs end to end and produces sane counts.

In [11]:
def load(stem):
    p=META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")
strokes=load("strokes")

VID="game_1"
df,pw = analyse(BASE/f"raw/videos/{VID}.mp4", video_id=VID)

gt=strokes[strokes.video_id==VID]
print("\n"+"="*66)
print(f"  shots detected : {len(df)}   ground truth {len(gt)}   "
      f"({len(df)/len(gt)-1:+.1%})")
print(f"  rallies        : {df.rally_id.nunique()}")
print(f"  mean rally len : {df.groupby('rally_id').size().mean():.1f} shots")
print(f"  abstained      : {df.abstain.sum()} ({df.abstain.mean():.0%})")
print(f"\n  predicted class mix vs ground truth:")
a=df.shot_class.value_counts(normalize=True)
b=gt.shot_class.value_counts(normalize=True)
for c in CLASSES:
    print(f"    {c:<9} {a.get(c,0):.3f}   truth {b.get(c,0):.3f}")
print(f"\n  pose_windows {pw.shape}   confidence "
      f"{df.class_confidence.mean():.3f} (calibrated, T={TEMP:.2f})")
print("="*66)
df.head(8)[["rally_id","shot_index","player","timestamp_s","shot_class",
            "class_confidence","peak_wrist_speed","backswing_amplitude"]]

game_1: 88,599 frames @ 120fps (12.3 min)


  activity:   0%|          | 0/174 [00:00<?, ?it/s]

  active: 43 regions, 57.0% of video


  pose:   0%|          | 0/50465 [00:00<?, ?it/s]

  contacts: 164
  164 shots, 29 rallies, 25 abstained  [4856s, 6.6x realtime]

  shots detected : 164   ground truth 161   (+1.9%)
  rallies        : 29
  mean rally len : 5.7 shots
  abstained      : 25 (15%)

  predicted class mix vs ground truth:
    serve     0.159   truth 0.149
    attack    0.384   truth 0.447
    control   0.085   truth 0.081
    defence   0.372   truth 0.323

  pose_windows (164, 97, 17, 2)   confidence 0.821 (calibrated, T=2.20)


,rally_id,shot_index,player,timestamp_s,shot_class,class_confidence,peak_wrist_speed,backswing_amplitude
0,0,0,left,0.491667,defence,0.790621,0.691758,1.073520
1,1,0,right,18.216667,serve,0.999745,0.144831,1.653661
2,1,1,left,18.708333,attack,0.769055,0.083459,0.787477
3,1,2,right,19.216667,attack,0.954192,0.600479,1.054690
4,2,0,right,76.883333,serve,0.998444,0.097438,1.568610
5,2,1,left,77.575000,control,0.807024,0.065270,1.484345
6,3,0,right,87.116667,serve,0.986293,0.096303,1.099862
7,3,1,left,87.658333,attack,0.839470,0.171044,1.405637


---
## Done

| artifact | contents |
|---|---|
| `derived/analysed/{vid}/shots.parquet` | one row per shot, schema-conformant |
| `derived/analysed/{vid}/pose_windows.npz` | 97×17×2 canonical window per shot — Phase 3's raw material |
| `derived/analysed/{vid}/pose_raw.npz` | cached pose, so re-running is instant |
| `derived/meta/calibration.json` | temperature + ECE before/after |

**Check:** shot count within ~10% of ground truth, rally lengths plausible (3–8 shots), ECE meaningfully lower after calibration.

To analyse any other video: `df, pw = analyse("/path/to/video.mp4")`.

Next: **Phase 2.5** — swap 2D pose for 3D lifted pose on the existing data and see whether accuracy holds. If it does, the sideline-camera constraint lifts.


In [12]:
# =============================================================================
# CELL 9 — PIPELINE FIXES
#
# FIX 1 — SPEED.  The activity gate did a random seek per sampled frame:
#     cap.set(CAP_PROP_POS_FRAMES, f); cap.read()
# On a 5.5 GB h264 file that is ~11,000 seeks, each forcing the decoder back
# to a keyframe and re-decoding forward. Replaced with grab()/retrieve():
# decode sequentially, convert only the sampled frames. grab() skips the
# YUV->BGR conversion, which is most of the per-frame cost.
#
# FIX 2 — ABSTENTION.  The floor was a per-video quantile, so it abstained on
# exactly ABSTAIN_Q of shots regardless of how confident the model actually
# was. That is backwards: a clean video should abstain rarely, a hard one
# often. Replaced with an ABSOLUTE threshold derived from calibration — the
# confidence below which accuracy drops under ACC_TARGET.
# =============================================================================

ACC_TARGET   = 0.70    # minimum accuracy for a shot to earn feedback
ACT_STRIDE   = 12      # was 8; players do not move much in 0.1 s
ACT_PAD_S    = 0.75    # was 1.0

import numpy as np, pandas as pd, torch, cv2, json, time, shutil
from pathlib import Path
from tqdm.auto import tqdm

# --- FIX 2: absolute abstention threshold from the calibration curve ---------
CAL = json.loads((META/"calibration.json").read_text())

def fit_abstain_floor(target=ACC_TARGET):
    """Lowest confidence at which accuracy still meets `target`."""
    d = np.load(BASE/"derived/clips/canonical.npz", allow_pickle=True)
    u = d["usable"]
    KP, VEL, VAL = d["kp"].astype(np.float32), d["vel"].astype(np.float32), d["valid"]
    TD = np.nan_to_num(d["table_dist"].astype(np.float32))
    X = np.nan_to_num(np.concatenate([
        KP[:,0].reshape(len(u),NF,-1), VEL[:,0].reshape(len(u),NF,-1),
        VAL[:,0].astype(np.float32), TD[...,None]], -1)).transpose(0,2,1)[u]
    y = pd.Series(d["shot_class"][u]).map({c:i for i,c in enumerate(CLASSES)}).values
    with torch.no_grad():
        lo,_ = CLS((torch.tensor(X, device=dev)-CMU)/CSD)
        pr = torch.softmax(lo/TEMP, 1).cpu().numpy()
    conf, corr = pr.max(1), (pr.argmax(1) == y).astype(float)
    # NOTE: these are TRAINING videos, so accuracy here is optimistic. The
    # SHAPE of the accuracy-vs-confidence curve still transfers, which is all
    # the threshold needs.
    rows = []
    for t in np.arange(0.30, 0.96, 0.05):
        m = conf >= t
        if m.sum() < 30: continue
        rows.append(dict(thr=round(t,2), kept=m.mean(), acc=corr[m].mean()))
    tab = pd.DataFrame(rows)
    ok = tab[tab.acc >= target]
    floor = float(ok.thr.min()) if len(ok) else 0.5
    return floor, tab

ABSTAIN_FLOOR, curve = fit_abstain_floor()
print("accuracy vs confidence threshold (training videos — optimistic):")
print(curve.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\n  abstain floor for >={ACC_TARGET:.2f} accuracy: {ABSTAIN_FLOOR:.2f}")
CAL["abstain_floor"] = ABSTAIN_FLOOR; CAL["acc_target"] = ACC_TARGET
(META/"calibration.json").write_text(json.dumps(CAL, indent=2))


# --- FIX 1: sequential activity gate ----------------------------------------
def activity_gate(path, nfr, mid_x, stride=ACT_STRIDE):
    """Sequential decode. grab() advances without converting to BGR;
    retrieve() converts only the frames we actually sample."""
    cap = cv2.VideoCapture(str(path))
    flags, idxs, buf, bidx, prev = [], [], [], [], None
    pb = tqdm(total=nfr, desc="  activity", leave=False)
    i = 0
    while i < nfr:
        if not cap.grab():
            break
        if i % stride == 0:
            ok, fr = cap.retrieve()
            if ok:
                buf.append(fr); bidx.append(i)
            if len(buf) >= 64:
                for j, d in enumerate(resolve(buf, mid_x)):
                    both = d["left"] is not None and d["right"] is not None
                    moving = True
                    if both and prev is not None:
                        h = max(d["left"][3]-d["left"][1], 1)
                        mv = max(abs((d["left"][0]+d["left"][2])/2 - prev[0]),
                                 abs((d["right"][0]+d["right"][2])/2 - prev[1]))/h
                        moving = mv > MOTION_THR
                    if both:
                        prev = ((d["left"][0]+d["left"][2])/2,
                                (d["right"][0]+d["right"][2])/2)
                    flags.append(both and moving); idxs.append(bidx[j])
                buf, bidx = [], []
        i += 1
        if i % 2000 == 0: pb.update(2000)
    if buf:
        for j, d in enumerate(resolve(buf, mid_x)):
            both = d["left"] is not None and d["right"] is not None
            flags.append(both); idxs.append(bidx[j])
    pb.close(); cap.release()

    pad = int(ACT_PAD_S*FPS); spans = []
    for k, f in enumerate(flags):
        if not f: continue
        s, e = max(0, idxs[k]-pad), min(nfr-1, idxs[k]+pad)
        if spans and s <= spans[-1][1]+1: spans[-1][1] = max(spans[-1][1], e)
        else: spans.append([s, e])
    return spans


# --- patched analyse() -------------------------------------------------------
_orig_analyse = analyse

def analyse(video_path, video_id=None, force=False, save_pose=True):
    """Same contract as before; uses the fixed gate and absolute floor."""
    video_path = Path(video_path); vid = video_id or video_path.stem
    outdir = ANALYSED/vid; outdir.mkdir(parents=True, exist_ok=True)
    cache = outdir/"pose_raw.npz"
    t0 = time.time()

    local = LOCAL/video_path.name
    if not local.exists(): shutil.copy(video_path, local)
    cap = cv2.VideoCapture(str(local))
    nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    table = find_table(cap, nfr); cap.release()
    mid_x = (table[0]+table[2])/2 if table is not None else W/2
    print(f"{vid}: {nfr:,} frames @ {fps:.0f}fps ({nfr/fps/60:.1f} min)")

    if cache.exists() and not force:
        raw = {k: v for k, v in np.load(cache, allow_pickle=True).items()}
        print(f"  cached pose: {len(raw['frame_idx']):,} frames")
    else:
        ta = time.time()
        spans = activity_gate(local, nfr, mid_x)
        cov = sum(e-s+1 for s, e in spans)/max(nfr, 1)
        print(f"  active: {len(spans)} regions, {cov:.1%} "
              f"[gate {time.time()-ta:.0f}s]")
        raw = extract_pose(local, spans, mid_x, nfr)
        if save_pose:
            np.savez_compressed(cache, **raw,
                table_box=table if table is not None else np.zeros(4, np.float32))
    st = build_stream(raw, table)

    prob = np.zeros(len(st["X"]), np.float32); sidep = np.zeros(len(st["X"]), np.float32)
    with torch.no_grad():
        for a, b in st["spans"]:
            x = torch.tensor(((st["X"][a:b]-DMU)/DSD).T[None],
                             dtype=torch.float32, device=dev)
            lc, ls = DET(x)
            prob[a:b] = torch.sigmoid(lc)[0].cpu().numpy()
            sidep[a:b] = torch.sigmoid(ls)[0].cpu().numpy()
    peaks = decode(prob)
    print(f"  contacts: {len(peaks)}")
    if not len(peaks):
        return pd.DataFrame(), np.zeros((0, NF, 17, 2), np.float16)

    sides = ["right" if sidep[i] >= 0.5 else "left" for i in peaks]
    wins, kps, vals = [], [], []
    for i, s in zip(peaks, sides):
        x, kp, val = window_at(st, i, s); wins.append(x); kps.append(kp); vals.append(val)
    with torch.no_grad():
        lo, lt = CLS((torch.tensor(np.stack(wins), device=dev)-CMU)/CSD)
        pr = torch.softmax(lo/TEMP, 1).cpu().numpy()
        pt = torch.softmax(lt/TEMP, 1).cpu().numpy()

    frames = st["fidx"][peaks]
    rid, shot_idx, cur, last = [], [], 0, None
    for f in frames:
        if last is not None and (f-last)/FPS > RALLY_GAP_S: cur += 1; k = 0
        else: k = 0 if last is None else shot_idx[-1]+1
        rid.append(cur); shot_idx.append(k); last = f
    rlen = pd.Series(rid).value_counts().to_dict()

    rows = []
    for n, (i, s) in enumerate(zip(peaks, sides)):
        pi = 0 if s == "left" else 1
        sel = np.clip(np.arange(i-PRE, i-PRE+NF), 0, len(st["X"])-1)
        km = kinematics(kps[n], vals[n], st["td"][sel, pi],
                        R_WRI if s == "left" else L_WRI)
        rows.append(dict(video_id=vid, rally_id=rid[n], shot_index=shot_idx[n],
            player=s, frame=int(frames[n]), timestamp_s=float(frames[n]/FPS),
            shot_class=CLASSES[int(pr[n].argmax())],
            class_confidence=float(pr[n].max()),
            class_proba=[float(z) for z in pr[n]],
            technique=TECHS[int(pt[n].argmax())],
            abstain=bool(pr[n].max() < ABSTAIN_FLOOR),      # FIX 2: absolute
            detect_confidence=float(prob[i]),
            rally_length=int(rlen[rid[n]]),
            pose_confidence=float(st["scores"][sel, pi][st["scores"][sel, pi] > 0].mean()
                                  if (st["scores"][sel, pi] > 0).any() else 0.),
            detected=float(st["detected"][sel, pi].mean()), **km))
    df = pd.DataFrame(rows)
    pose_windows = np.stack(kps).astype(np.float16)

    df.to_parquet(outdir/"shots.parquet", index=False)
    np.savez_compressed(outdir/"pose_windows.npz",
        stroke_id=df.apply(lambda r: f"{vid}_{r.frame:07d}", axis=1).values.astype(str),
        pose_window=pose_windows, valid=np.stack(vals))
    el = time.time()-t0
    print(f"  {len(df)} shots, {df.rally_id.nunique()} rallies, "
          f"{df.abstain.sum()} abstained ({df.abstain.mean():.0%})  "
          f"[{el:.0f}s, {el/(nfr/fps):.1f}x realtime]")
    return df, pose_windows

print(f"\npatched. abstain floor {ABSTAIN_FLOOR:.2f} (absolute), "
      f"gate stride {ACT_STRIDE}, pad {ACT_PAD_S}s")
print("Re-run with force=True to rebuild the pose cache and time the new gate.")

accuracy vs confidence threshold (training videos — optimistic):
  thr  kept   acc
0.300 1.000 0.963
0.350 1.000 0.963
0.400 0.999 0.963
0.450 0.993 0.966
0.500 0.981 0.970
0.550 0.965 0.974
0.600 0.944 0.979
0.650 0.914 0.985
0.700 0.873 0.991
0.750 0.818 0.992
0.800 0.708 0.992
0.850 0.547 0.994
0.900 0.429 0.993
0.950 0.318 0.993

  abstain floor for >=0.70 accuracy: 0.30

patched. abstain floor 0.30 (absolute), gate stride 12, pad 0.75s
Re-run with force=True to rebuild the pose cache and time the new gate.


In [13]:
# =============================================================================
# CELL 10 — ABSTENTION FLOOR, CORRECTED
#
# The previous fit produced floor = 0.30 (the bottom of the search range)
# because accuracy was 0.963 at EVERY threshold. That curve is flat, so the
# floor abstains on nothing.
#
# Cause: it was fitted on the FINAL model's predictions over its own training
# data. That model memorised those strokes (final training loss 0.0042), so it
# is never unreliable on them and there is no signal about where reliability
# breaks down. A threshold whose purpose is detecting unreliability cannot be
# fitted where the model is never unreliable.
#
# Correct source: the 7-fold LOVO out-of-fold predictions — genuinely
# held-out, and the same predictions the temperature was fitted on. Cell 4
# computed them and discarded them; this recomputes and caches them.
#
# ~10 min the first time, instant afterwards.
# =============================================================================

ACC_TARGET = 0.70
OOF_PATH   = META/"oof_logits.npz"

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import math, json

if OOF_PATH.exists():
    z = np.load(OOF_PATH)
    OOF, OOF_Y = z["logits"], z["y"]
    print(f"loaded cached OOF logits {OOF.shape}")
else:
    print("computing 7-fold LOVO out-of-fold predictions ...")
    d = np.load(BASE/"derived/clips/canonical.npz", allow_pickle=True)
    u = d["usable"]
    KP, VEL, VAL = d["kp"].astype(np.float32), d["vel"].astype(np.float32), d["valid"]
    TD = np.nan_to_num(d["table_dist"].astype(np.float32))
    X = np.nan_to_num(np.concatenate([
        KP[:,0].reshape(len(u),NF,-1), VEL[:,0].reshape(len(u),NF,-1),
        VAL[:,0].astype(np.float32), TD[...,None]], -1)).transpose(0,2,1)[u]
    y = pd.Series(d["shot_class"][u]).map({c:i for i,c in enumerate(CLASSES)}).values
    tc = pd.Series(d["technique"][u]).map({t:i for i,t in enumerate(TECHS)}
                                          ).fillna(-1).astype(int).values
    fc = d["fold"][u]

    def focal(lg, tg, w=None, g=2.0):
        m = tg >= 0
        if m.sum() == 0: return lg.sum()*0.
        lg, tg = lg[m], tg[m]
        ce = F.cross_entropy(lg, tg, weight=w, reduction="none")
        pt = torch.exp(-F.cross_entropy(lg, tg, reduction="none"))
        return ((1-pt)**g*ce).mean()

    OOF = np.zeros((len(y), 4), np.float32)
    for f in sorted(set(fc)):
        tr, va = fc != f, fc == f
        xt = torch.tensor(X[tr], device=dev)
        mu, sd = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True)+1e-6
        xt = (xt-mu)/sd
        yt = torch.tensor(y[tr], device=dev); tt = torch.tensor(tc[tr], device=dev)
        cnt = np.bincount(y[tr], minlength=4).clip(1)
        cw = torch.tensor(len(y[tr])/(4*cnt), dtype=torch.float32, device=dev)
        p = (1./cnt)[y[tr]]; p = p/p.sum()
        net = ClsNet(X.shape[1]).to(dev)
        opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
        steps = 60*max(1, math.ceil(tr.sum()/64))
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
        net.train()
        for _ in range(steps):
            b = np.random.choice(tr.sum(), 64, p=p)
            xb = xt[b]
            sh = torch.randint(-6, 7, (64,), device=dev)
            ix = (torch.arange(NF, device=dev)[None]+sh[:,None]).clamp(0, NF-1)
            xb = torch.gather(xb, 2, ix[:,None].expand(-1, xb.shape[1], -1))
            s_, t_ = net(xb + torch.randn_like(xb)*0.01)
            loss = focal(s_, yt[b], cw) + 0.2*focal(t_, tt[b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
        net.eval()
        with torch.no_grad():
            lo, _ = net((torch.tensor(X[va], device=dev)-mu)/sd)
            OOF[va] = lo.cpu().numpy()
        print(f"  fold {f} done")
    OOF_Y = y
    np.savez_compressed(OOF_PATH, logits=OOF, y=OOF_Y)
    print(f"-> {OOF_PATH}")

# --- the curve, on held-out predictions --------------------------------------
pr = torch.softmax(torch.tensor(OOF)/TEMP, 1).numpy()
conf, corr = pr.max(1), (pr.argmax(1) == OOF_Y).astype(float)
print(f"\nheld-out accuracy overall: {corr.mean():.3f}  "
      f"(training-data version showed 0.963 — that was the bug)")

rows = []
for t in np.arange(0.30, 0.96, 0.05):
    m = conf >= t
    if m.sum() < 30: continue
    rows.append(dict(thr=round(t,2), kept=m.mean(), acc=corr[m].mean(),
                     n=int(m.sum())))
curve = pd.DataFrame(rows)
print("\naccuracy vs confidence threshold (HELD-OUT):")
print(curve.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

ok = curve[curve.acc >= ACC_TARGET]
ABSTAIN_FLOOR = float(ok.thr.min()) if len(ok) else float(curve.thr.max())
kept = float(curve.loc[curve.thr == ABSTAIN_FLOOR, "kept"].iloc[0])
acc_at = float(curve.loc[curve.thr == ABSTAIN_FLOOR, "acc"].iloc[0])

print("\n" + "=" * 70)
print(f"  abstain floor : {ABSTAIN_FLOOR:.2f}")
print(f"  keeps         : {kept:.1%} of shots at {acc_at:.3f} accuracy")
print(f"  abstains on   : {1-kept:.1%}")
if ABSTAIN_FLOOR <= 0.30:
    print("\n  !! Still at the range floor — the model already exceeds the")
    print("     target everywhere. Raise ACC_TARGET if you want stricter gating.")
print("=" * 70)

# per-class abstention: which classes get withheld most?
cls_pred = pr.argmax(1)
print("\nabstention by predicted class:")
for i, c in enumerate(CLASSES):
    m = cls_pred == i
    if not m.sum(): continue
    ab = (conf[m] < ABSTAIN_FLOOR).mean()
    a_keep = corr[m & (conf >= ABSTAIN_FLOOR)].mean() if (m & (conf >= ABSTAIN_FLOOR)).sum() else np.nan
    print(f"  {c:<9} n={m.sum():>4}  abstained {ab:.1%}   "
          f"accuracy when kept {a_keep:.3f}")
print("""
  Expect `defence` to abstain most — it is the weakest class (LOVO F1 0.471)
  and withholding feedback there is the correct behaviour, not a failure.""")

CAL = json.loads((META/"calibration.json").read_text())
CAL.update(abstain_floor=ABSTAIN_FLOOR, acc_target=ACC_TARGET,
           fitted_on="lovo_oof", kept_fraction=kept, acc_at_floor=acc_at)
(META/"calibration.json").write_text(json.dumps(CAL, indent=2))
print(f"\n-> {META/'calibration.json'}   ABSTAIN_FLOOR={ABSTAIN_FLOOR:.2f}")
print("Re-run analyse(..., force=True) to apply.")

computing 7-fold LOVO out-of-fold predictions ...
  fold A done
  fold B done
  fold C done
  fold D done
  fold E done
  fold F done
  fold G done
-> /content/drive/MyDrive/tt_coach/derived/meta/oof_logits.npz

held-out accuracy overall: 0.757  (training-data version showed 0.963 — that was the bug)

accuracy vs confidence threshold (HELD-OUT):
  thr  kept   acc    n
0.300 0.999 0.758 1431
0.350 0.992 0.761 1420
0.400 0.973 0.770 1394
0.450 0.935 0.786 1339
0.500 0.876 0.809 1254
0.550 0.812 0.826 1163
0.600 0.756 0.841 1082
0.650 0.685 0.859  981
0.700 0.626 0.880  897
0.750 0.564 0.902  807
0.800 0.498 0.930  713
0.850 0.423 0.949  606
0.900 0.350 0.968  501
0.950 0.251 0.981  359

  abstain floor : 0.30
  keeps         : 99.9% of shots at 0.758 accuracy
  abstains on   : 0.1%

  !! Still at the range floor — the model already exceeds the
     target everywhere. Raise ACC_TARGET if you want stricter gating.

abstention by predicted class:
  serve     n= 269  abstained 0.0%   accurac

In [14]:
# =============================================================================
# CELL 11 — PER-CLASS ABSTENTION THRESHOLDS
#
# A single global floor cannot serve this model. Held-out precision by
# predicted class:
#
#     serve    0.974      needs no gating at all
#     attack   0.801      light gating
#     control  0.741      moderate gating
#     defence  0.403      heavy gating, possibly total suppression
#
# One threshold either wastes reliable serve predictions or lets through
# defence predictions that are wrong 6 times in 10. For coaching, a wrong
# class means comparing a shot against the wrong reference distribution and
# giving confidently wrong advice — worse than saying nothing.
#
# This fits ONE THRESHOLD PER CLASS on the held-out LOVO predictions, and
# reports honestly when a class cannot reach the target at any confidence.
# =============================================================================

ACC_TARGET = 0.85      # required precision for a class to earn feedback
MIN_KEEP   = 0.15      # a class kept below this rate is not worth gating for
MIN_N      = 25        # minimum samples to trust a threshold estimate

import numpy as np, pandas as pd, torch, json

z = np.load(META/"oof_logits.npz")
pr = torch.softmax(torch.tensor(z["logits"])/TEMP, 1).numpy()
y  = z["y"]
pred, conf = pr.argmax(1), pr.max(1)
corr = (pred == y).astype(float)

print(f"held-out: {len(y)} shots, overall accuracy {corr.mean():.3f}\n")

# --- precision vs threshold, per class ---------------------------------------
grid = np.arange(0.25, 0.99, 0.025)
tab = []
for i, c in enumerate(CLASSES):
    m = pred == i
    for t in grid:
        k = m & (conf >= t)
        if k.sum() < MIN_N: continue
        tab.append(dict(cls=c, thr=round(float(t), 3),
                        precision=float(corr[k].mean()),
                        kept=float(k.sum()/max(m.sum(), 1)),
                        n=int(k.sum())))
tab = pd.DataFrame(tab)

print("precision vs threshold, per predicted class")
print("=" * 72)
for c in CLASSES:
    g = tab[tab.cls == c]
    if not len(g):
        print(f"\n{c}: too few samples"); continue
    show = g[g.thr.isin([0.25, 0.4, 0.55, 0.7, 0.8, 0.9, 0.95])]
    print(f"\n{c}  (n={int((pred == CLASSES.index(c)).sum())})")
    print(show[["thr", "precision", "kept", "n"]].to_string(
        index=False, float_format=lambda x: f"{x:.3f}"))

# --- pick a threshold per class ----------------------------------------------
THRESHOLDS, report = {}, []
for i, c in enumerate(CLASSES):
    g = tab[tab.cls == c]
    ok = g[(g.precision >= ACC_TARGET) & (g.kept >= MIN_KEEP)]
    n_tot = int((pred == i).sum())
    if len(ok):
        r = ok.loc[ok.thr.idxmin()]
        THRESHOLDS[c] = float(r.thr)
        report.append(dict(cls=c, threshold=r.thr, precision=r.precision,
                           kept=r.kept, n_kept=int(r.n), n_total=n_tot,
                           status="feedback"))
    else:
        best = g.precision.max() if len(g) else np.nan
        THRESHOLDS[c] = 1.01          # never passes -> always abstain
        report.append(dict(cls=c, threshold=np.nan, precision=best,
                           kept=0.0, n_kept=0, n_total=n_tot,
                           status="SUPPRESS"))

rep = pd.DataFrame(report)
print("\n" + "=" * 72)
print(f"PER-CLASS THRESHOLDS   (target precision {ACC_TARGET}, "
      f"min coverage {MIN_KEEP:.0%})")
print("=" * 72)
print(rep.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

sup = rep[rep.status == "SUPPRESS"]
if len(sup):
    print(f"\n  SUPPRESSED: {list(sup.cls)}")
    print(f"""
  These classes never reach {ACC_TARGET:.0%} precision at any confidence, so
  Phase 3 must give NO stroke-specific feedback on them. This is a real
  limitation of pose-only classification, not a threshold to tune away.
  A shot predicted as one of these can still be counted in rally statistics —
  it just cannot be coached as that stroke type.""")

# --- what this costs in coverage ---------------------------------------------
keep_mask = np.array([conf[k] >= THRESHOLDS[CLASSES[pred[k]]]
                      for k in range(len(pred))])
print("\n" + "=" * 72)
print("OVERALL EFFECT")
print("=" * 72)
print(f"  coachable shots : {keep_mask.mean():.1%} "
      f"({int(keep_mask.sum())} of {len(pred)})")
print(f"  precision on kept: {corr[keep_mask].mean():.3f}"
      if keep_mask.sum() else "  nothing kept")
print(f"  vs no gating     : {corr.mean():.3f} over 100%")
print(f"\n  Trade: {1-keep_mask.mean():.0%} of shots go uncoached, and the rest "
      f"carry {corr[keep_mask].mean():.0%} reliable labels.")

# --- sensitivity to the target -----------------------------------------------
print("\n" + "=" * 72)
print("SENSITIVITY — coverage at different precision targets")
print("=" * 72)
rows = []
for tgt in (0.75, 0.80, 0.85, 0.90, 0.95):
    th, n_sup = {}, 0
    for i, c in enumerate(CLASSES):
        g = tab[tab.cls == c]
        ok = g[(g.precision >= tgt) & (g.kept >= MIN_KEEP)]
        if len(ok): th[c] = float(ok.thr.min())
        else: th[c] = 1.01; n_sup += 1
    km = np.array([conf[k] >= th[CLASSES[pred[k]]] for k in range(len(pred))])
    rows.append(dict(target=tgt, coachable=km.mean(),
                     actual_precision=corr[km].mean() if km.sum() else np.nan,
                     suppressed_classes=n_sup))
print(pd.DataFrame(rows).to_string(index=False,
                                   float_format=lambda x: f"{x:.3f}"))

CAL = json.loads((META/"calibration.json").read_text())
CAL.update(per_class_thresholds=THRESHOLDS, acc_target=ACC_TARGET,
           min_keep=MIN_KEEP, fitted_on="lovo_oof",
           coachable_fraction=float(keep_mask.mean()),
           precision_on_kept=float(corr[keep_mask].mean()) if keep_mask.sum() else None)
(META/"calibration.json").write_text(json.dumps(CAL, indent=2))
print(f"\n-> {META/'calibration.json'}")
print(f"   thresholds: { {k: round(v,3) for k,v in THRESHOLDS.items()} }")


# --- patch analyse() to use them ---------------------------------------------
def apply_per_class_abstain(df):
    """Recompute `abstain` on an existing shots.parquet using per-class rules."""
    df = df.copy()
    df["abstain"] = [row.class_confidence < THRESHOLDS[row.shot_class]
                     for row in df.itertuples()]
    return df

print("""
To apply: re-run analyse(..., force=False) after replacing the abstain line
with

    abstain=bool(pr[n].max() < THRESHOLDS[CLASSES[int(pr[n].argmax())]]),

or call apply_per_class_abstain(df) on an already-analysed DataFrame.""")

held-out: 1432 shots, overall accuracy 0.757

precision vs threshold, per predicted class

serve  (n=269)
  thr  precision  kept   n
0.250      0.974 1.000 269
0.400      0.974 1.000 269
0.550      0.985 0.974 262
0.700      0.992 0.944 254
0.800      0.996 0.929 250
0.900      0.996 0.892 240
0.950      0.996 0.833 224

attack  (n=657)
  thr  precision  kept   n
0.250      0.801 1.000 657
0.400      0.813 0.979 643
0.550      0.870 0.843 554
0.700      0.917 0.697 458
0.800      0.937 0.580 381
0.900      0.971 0.370 243
0.950      0.985 0.199 131

control  (n=274)
  thr  precision  kept   n
0.250      0.741 1.000 274
0.400      0.759 0.953 261
0.550      0.766 0.719 197
0.700      0.757 0.391 107
0.800      0.804 0.168  46

defence  (n=232)
  thr  precision  kept   n
0.250      0.401 1.000 232
0.400      0.407 0.953 221
0.550      0.467 0.647 150
0.700      0.462 0.336  78
0.800      0.556 0.155  36

PER-CLASS THRESHOLDS   (target precision 0.85, min coverage 15%)
    cls  threshold 